# Lab 03 — Signature Authenticity Verification (Siamese Network)

> **GraphoLab** | Forensic Graphology Laboratory

**Technique:** Siamese Neural Network (SigNet architecture)  
**Task:** Compare two signatures and determine genuine vs. forged  
**Forensic use case:** Bank cheques, contracts, wills, identity documents

---

## How a Siamese Network Works

A **Siamese network** consists of two **identical, weight-sharing** branches. Each branch encodes one input image into a feature vector (embedding). The distance between the two embeddings determines the verdict:

```
Reference Signature ──► [Encoder] ──► embedding_ref ─┐
                                                       ├──► distance ──► genuine / forged
Questioned Signature ──► [Encoder] ──► embedding_q  ─┘
```

- **Small distance** → similar style → **genuine**
- **Large distance** → different style → **forged**

The encoder is trained with **contrastive loss** on pairs of genuine and forged signatures.

**Reference:** Li, G. et al. — SigNet: Convolutional Siamese Network for writer-independent offline signature verification.


## Setup


In [ ]:
# !pip install torch torchvision Pillow matplotlib scikit-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageOps
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## SigNet Encoder Architecture

We implement the SigNet CNN encoder as described in the original paper.


In [ ]:
class SigNetEncoder(nn.Module):
    """SigNet CNN encoder for signature embeddings.

    Architecture from: Dey et al. (2017) arXiv:1707.02131
    Input: grayscale signature image, resized to 155x220.
    Output: 128-dimensional L2-normalised feature vector.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 96, kernel_size=11, stride=1, padding=0),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            # Block 2
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(5, alpha=1e-4, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout2d(0.3),
            # Block 3
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            # Block 4
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout2d(0.3),
        )
        self.fc = nn.Sequential(
            nn.Linear(256 * 6 * 9, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 128),
        )

    def forward_once(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return F.normalize(x, p=2, dim=1)  # L2-normalise

    def forward(self, x1: torch.Tensor, x2: torch.Tensor):
        return self.forward_once(x1), self.forward_once(x2)


model = SigNetEncoder().to(DEVICE)
print(f"SigNet encoder ready — {sum(p.numel() for p in model.parameters()):,} parameters")

## Image Preprocessing


In [ ]:
# SigNet expects grayscale images at 155×220 pixels with white background
TRANSFORM = T.Compose([
    T.Grayscale(),
    T.Resize((155, 220)),
    T.ToTensor(),           # [0, 1]
    T.Normalize([0.5], [0.5])  # → [-1, 1]
])


def preprocess(img: Image.Image) -> torch.Tensor:
    """Preprocess a PIL signature image for SigNet."""
    img = img.convert('RGB')
    # Invert if background is dark (signatures are typically dark on white)
    arr = np.array(img.convert('L'))
    if arr.mean() < 128:
        img = ImageOps.invert(img)
    return TRANSFORM(img).unsqueeze(0).to(DEVICE)  # (1, 1, 155, 220)


def load_signature(path: str | Path) -> Image.Image:
    return Image.open(path).convert('RGB')

## Similarity Score


In [ ]:
THRESHOLD = 0.35  # cosine distance threshold: < THRESHOLD → genuine


def verify_signatures(img1: Image.Image, img2: Image.Image) -> dict:
    """Compare two signature images and return verdict + scores."""
    model.eval()
    t1, t2 = preprocess(img1), preprocess(img2)
    with torch.no_grad():
        emb1, emb2 = model(t1, t2)

    cosine_sim = F.cosine_similarity(emb1, emb2).item()   # [-1, 1]
    cosine_dist = 1.0 - cosine_sim                        # [0, 2]
    euclidean_dist = torch.dist(emb1, emb2).item()

    # Normalise cosine distance to [0, 1] confidence
    confidence = max(0.0, min(1.0, 1.0 - cosine_dist / 2.0))

    verdict = "GENUINE" if cosine_dist < THRESHOLD else "FORGED"

    return {
        "verdict": verdict,
        "confidence": confidence,
        "cosine_similarity": cosine_sim,
        "cosine_distance": cosine_dist,
        "euclidean_distance": euclidean_dist,
    }


def show_verification(img_ref: Image.Image, img_q: Image.Image, result: dict) -> None:
    """Display the two signatures side by side with the verdict."""
    fig = plt.figure(figsize=(12, 4))
    gs = gridspec.GridSpec(1, 3, figure=fig, width_ratios=[2, 2, 1.5])

    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_ref, cmap='gray')
    ax1.set_title("Reference Signature", fontsize=11)
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img_q, cmap='gray')
    ax2.set_title("Questioned Signature", fontsize=11)
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[2])
    ax3.axis('off')
    color = 'green' if result['verdict'] == 'GENUINE' else 'red'
    verdict_text = (
        f"Verdict: {result['verdict']}\n\n"
        f"Confidence: {result['confidence']:.1%}\n"
        f"Cosine sim: {result['cosine_similarity']:.4f}\n"
        f"Cosine dist: {result['cosine_distance']:.4f}\n"
        f"(threshold: {THRESHOLD})"
    )
    ax3.text(0.1, 0.5, verdict_text, fontsize=11, va='center',
             color=color, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    plt.suptitle("Signature Verification", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Demo — Synthetic Signature Pairs

> **Note:** The model weights here are randomly initialised (for demo purposes without downloading pre-trained weights). In a real forensic deployment, load weights trained on a signature dataset such as CEDAR or SigComp'11. See the reference below for pre-trained weights.
>
> Pre-trained weights: https://github.com/luizgh/sigver


In [ ]:
def make_synthetic_signature(text: str = "J. Smith", noise: float = 0.0,
                              size=(220, 155)) -> Image.Image:
    """Create a simple synthetic 'signature' image for demo purposes."""
    from PIL import ImageDraw, ImageFilter
    img = Image.new('RGB', size, color='white')
    draw = ImageDraw.Draw(img)
    # Draw a simple cursive-like curve
    draw.text((20, 60), text, fill='black')
    # Add underline stroke
    draw.line([(20, 90), (180, 90)], fill='black', width=2)
    draw.arc([(160, 70), (200, 100)], start=0, end=180, fill='black', width=2)
    if noise > 0:
        arr = np.array(img, dtype=np.float32)
        arr += np.random.normal(0, noise * 255, arr.shape)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)
    return img


samples_dir = Path("../data/samples")

# Try loading real samples; fall back to synthetic
ref_path = samples_dir / "signature_genuine_01.png"
genuine_path = samples_dir / "signature_genuine_02.png"
forged_path = samples_dir / "signature_forged_01.png"

if ref_path.exists() and genuine_path.exists() and forged_path.exists():
    sig_ref = load_signature(ref_path)
    sig_genuine = load_signature(genuine_path)
    sig_forged = load_signature(forged_path)
    print("Loaded real signature samples.")
else:
    sig_ref = make_synthetic_signature("J. Smith")
    sig_genuine = make_synthetic_signature("J. Smith", noise=0.02)  # slight variation
    sig_forged = make_synthetic_signature("Forger X", noise=0.05)   # different style
    print("Using synthetic signature samples (add real PNGs to data/samples/ for better results).")

In [ ]:
# Test 1: Reference vs. Genuine copy
result_genuine = verify_signatures(sig_ref, sig_genuine)
print("Reference vs. Genuine:")
for k, v in result_genuine.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
show_verification(sig_ref, sig_genuine, result_genuine)

In [ ]:
# Test 2: Reference vs. Forged
result_forged = verify_signatures(sig_ref, sig_forged)
print("Reference vs. Forged:")
for k, v in result_forged.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
show_verification(sig_ref, sig_forged, result_forged)

## Loading Pre-trained Weights (Optional)

For real forensic use, download pre-trained SigNet weights:


In [ ]:
# Uncomment and adjust if you have pre-trained weights:
#
# weights_path = Path("../data/signet_cedar.pth")
# if weights_path.exists():
#     state = torch.load(weights_path, map_location=DEVICE)
#     model.load_state_dict(state)
#     print(f"Loaded weights from {weights_path}")
#
# Pre-trained weights and training code:
# https://github.com/luizgh/sigver

print("Pre-trained weights not loaded. Using random initialisation for demo.")
print("See https://github.com/luizgh/sigver for CEDAR-trained weights.")

## Forensic Notes

- The **threshold** (default 0.35) must be calibrated on a representative dataset. A lower threshold reduces false acceptances (genuine labelled as forged); a higher threshold reduces false rejections.
- Forensic signature verification should always be accompanied by **multiple reference samples** (at least 10–15 genuine signatures per person).
- Skilled forgeries may pass automated checks — AI is a screening tool, not a final arbiter.
- Consider also online (dynamic) signature features (pen speed, pressure over time) when available, as they are harder to forge.

---

**Next lab →** [04 — Signature Detection in Documents (YOLOv8)](04_signature_detection_yolo.ipynb)
